(1) Data Preprocessing & Cleaning

In [1]:
import pandas as pd
import numpy as np

file_path = 'multi_platform_social_sentiment_evolution.csv'

try:
    df = pd.read_csv(file_path)
    print("--- Đọc file thành công! ---")
except FileNotFoundError:
    print("Không tìm thấy file. Hãy kiểm tra lại đường dẫn.")

print("\nThông tin Dataset:")
print(df.info())


--- Đọc file thành công! ---

Thông tin Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 31 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   post_id                           150000 non-null  object 
 1   platform                          150000 non-null  object 
 2   timestamp                         150000 non-null  object 
 3   date                              150000 non-null  object 
 4   hour_of_day                       150000 non-null  int64  
 5   day_of_week                       150000 non-null  int64  
 6   is_weekend                        150000 non-null  int64  
 7   user_id                           150000 non-null  object 
 8   followers                         150000 non-null  int64  
 9   account_age_days                  150000 non-null  int64  
 10  verified                          150000 non-null  int64  
 11  top

1.1 parse datetime

In [2]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Extract additional time features (optional)
df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.day
df['month'] = df['timestamp'].dt.month

1.2 Feature Transformation

In [3]:
df['log_followers'] = np.log1p(df['followers'])

1.3 DEFINE FEATURE GROUPS

In [4]:
# =========================
# 6. DEFINE FEATURE GROUPS
# =========================

categorical_cols = [
    'platform',
    'topic',
    'language',
    'media_type',
    'sentiment_category',
    'location'
]

numerical_cols = [
    'log_followers',
    'account_age_days',
    'content_length',
    'num_hashtags',
    'sentiment_positive',
    'sentiment_negative',
    'sentiment_neutral',
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'hours_since_post',
    'toxicity_score'
]


1.4 SCALING + ENCODING PIPELINE

In [5]:
# =========================
# 7. SCALING + ENCODING PIPELINE
# =========================
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

1.5 APPLY TRANSFORM

In [6]:
# =========================
# 8. APPLY TRANSFORM
# =========================

X = preprocessor.fit_transform(df)

print("Processed feature shape:", X.shape)



Processed feature shape: (150000, 58)


In [7]:
# =========================
# 9. SAVE PREPROCESSOR (IMPORTANT)
# =========================
import joblib
joblib.dump(preprocessor, "preprocessor.pkl")

print("Preprocessing done ✅")

Preprocessing done ✅


1.6 DEFINE TARGET

In [ ]:
# Target chính
df['target_engagement'] = df['engagement_rate_per_1k_followers']

1.7 CLASSIFICATION LABEL

In [9]:
# dùng percentile để chia high / low engagement
threshold = df['target_engagement'].quantile(0.8)

df['label'] = (df['target_engagement'] > threshold).astype(int)

print("Threshold:", threshold)
print(df['label'].value_counts())

Threshold: 40.0
label
0    120012
1     29988
Name: count, dtype: int64


1.8 Drop column

In [10]:
drop_cols = [
    'post_id',
    'timestamp',
]

df = df.drop(columns=drop_cols)

In [11]:
leakage_cols = [
    'likes',
    'shares',
    'comments',
    'total_engagement',
    'engagement_rate_per_1k_followers',
    'viral_coefficient'
]

df = df.drop(columns=leakage_cols)

print("Remaining columns:", df.columns)

Remaining columns: Index(['platform', 'date', 'hour_of_day', 'day_of_week', 'is_weekend',
       'user_id', 'followers', 'account_age_days', 'verified', 'topic',
       'language', 'content_length', 'media_type', 'num_hashtags',
       'sentiment_category', 'sentiment_positive', 'sentiment_negative',
       'sentiment_neutral', 'views', 'hours_since_post',
       'cross_platform_spread', 'toxicity_score', 'location', 'hour', 'day',
       'month', 'log_followers', 'target_engagement', 'label'],
      dtype='object')


In [12]:
# ---- Regression ----
y_reg = df['target_engagement']

# ---- Classification ----
y_clf = df['label']

# X = tất cả trừ target + label
X_df = df.drop(columns=['target_engagement', 'label'])

print("X shape:", X_df.shape)
print("y_reg shape:", y_reg.shape)
print("y_clf shape:", y_clf.shape)

X shape: (150000, 27)
y_reg shape: (150000,)
y_clf shape: (150000,)


2. Train split

In [ ]:
# preprocess
X_processed = preprocessor.fit_transform(X_df)

# split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X_processed, y_reg, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (120000, 27)
Test shape: (30000, 27)


BASELINE 1: MEAN PREDICTION

In [15]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# predict mean
mean_value = y_train_reg.mean()
y_pred_mean = np.full_like(y_test_reg, mean_value)

# evaluate
mae = mean_absolute_error(y_test_reg, y_pred_mean)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_mean))

print("Baseline 1 (Mean):")
print("MAE:", mae)
print("RMSE:", rmse)

Baseline 1 (Mean):
MAE: 82.18655074757777
RMSE: 312.13167326373366


BASELINE 2: LINEAR REGRESSION

In [18]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

model_lr = LinearRegression()
model_lr.fit(X_train, y_train_reg)

y_pred_lr = model_lr.predict(X_test)

print("\nBaseline 2 (Linear Regression):")
print("MAE:", mean_absolute_error(y_test_reg, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test_reg, y_pred_lr)))
print("R2:", r2_score(y_test_reg, y_pred_lr))


Baseline 2 (Linear Regression):
MAE: 82.08769790129003
RMSE: 312.2093330997923
R2: -0.0004995211783083153


In [ ]:
# 1. Transform trước
X = preprocessor.fit_transform(df)

# 2. Split sau khi transform
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_reg, y_test = train_test_split(
    X, df['target_engagement'], test_size=0.2, random_state=42
)

# 3. Train model
model_lr.fit(X_train, y_train_reg)

BASELINE 3: LOGISTIC REGRESSION (CLASSIFICATION)

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

model_log = LogisticRegression(max_iter=1000)
model_log.fit(X_train, y_train_clf)

y_pred_clf = model_log.predict(X_test)
y_prob = model_log.predict_proba(X_test)[:, 1]

print("\nBaseline 3 (Logistic Regression):")
print("Accuracy:", accuracy_score(y_test_clf, y_pred_clf))
print("F1-score:", f1_score(y_test_clf, y_pred_clf))
print("ROC-AUC:", roc_auc_score(y_test_clf, y_prob))


Baseline 3 (Logistic Regression):
Accuracy: 0.8017
F1-score: 0.0
ROC-AUC: 0.5630579913185125


A. TRAIN XGBOOST (FIX IMBALANCE)

In [20]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# =========================
# 1. CALCULATE CLASS WEIGHT
# =========================
neg = (y_train_clf == 0).sum()
pos = (y_train_clf == 1).sum()

scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)


scale_pos_weight: 3.99188818170473


In [21]:
from xgboost import XGBClassifier
# =========================
# 2. TRAIN MODEL
# =========================
model_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

model_xgb.fit(X_train, y_train_clf)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [22]:
# =========================
# 3. PREDICT
# =========================
y_pred = model_xgb.predict(X_test)
y_prob = model_xgb.predict_proba(X_test)[:, 1]

print("\nBaseline 4 (XGBoost):")
print("Accuracy:", accuracy_score(y_test_clf, y_pred))
print("F1-score:", f1_score(y_test_clf, y_pred))
print("ROC-AUC:", roc_auc_score(y_test_clf, y_prob))



Baseline 4 (XGBoost):
Accuracy: 0.5292333333333333
F1-score: 0.32097697004663683
ROC-AUC: 0.5624138314978524


Importance features

In [24]:
feature_names = preprocessor.get_feature_names_out()

importances = model_xgb.feature_importances_

feat_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_importance.head(10))

                             feature  importance
0                 num__log_followers    0.048383
50  cat__sentiment_category_Positive    0.046746
16             cat__platform_Twitter    0.019107
13           cat__platform_Instagram    0.018527
14              cat__platform_Reddit    0.018370
19               cat__topic_Business    0.018112
25                   cat__topic_Food    0.017979
33              cat__language_Arabic    0.017728
40              cat__language_Korean    0.017635
4            num__sentiment_positive    0.017629
